<a href="https://colab.research.google.com/github/Dilraj112358/4401Project2/blob/main/Stanley_Project_2_NN_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv('application_train.csv')

y = df['TARGET']
X = df.drop(columns=["TARGET", "SK_ID_CURR"])


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# One-hot encode these
categorical_cols = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY", "NAME_TYPE_SUITE", "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE", "OCCUPATION_TYPE",
    "WEEKDAY_APPR_PROCESS_START", "ORGANIZATION_TYPE",
    "FONDKAPREMONT_MODE", "HOUSETYPE_MODE",
    "WALLSMATERIAL_MODE", "EMERGENCYSTATE_MODE",
    "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY"
]

# Impute missing values and standard scale these
scale_cols = [
    "CNT_CHILDREN", "AMT_INCOME_TOTAL", "AMT_CREDIT",
    "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "REGION_POPULATION_RELATIVE", "DAYS_BIRTH",
    "DAYS_EMPLOYED", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH",
    "OWN_CAR_AGE", "CNT_FAM_MEMBERS", "HOUR_APPR_PROCESS_START",
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "APARTMENTS_AVG", "BASEMENTAREA_AVG",
    "YEARS_BEGINEXPLUATATION_AVG", "YEARS_BUILD_AVG",
    "COMMONAREA_AVG", "ELEVATORS_AVG", "ENTRANCES_AVG",
    "FLOORSMAX_AVG", "FLOORSMIN_AVG", "LANDAREA_AVG",
    "LIVINGAPARTMENTS_AVG", "LIVINGAREA_AVG",
    "NONLIVINGAPARTMENTS_AVG", "NONLIVINGAREA_AVG",
    "APARTMENTS_MODE", "BASEMENTAREA_MODE",
    "YEARS_BEGINEXPLUATATION_MODE", "YEARS_BUILD_MODE",
    "COMMONAREA_MODE", "ELEVATORS_MODE", "ENTRANCES_MODE",
    "FLOORSMAX_MODE", "FLOORSMIN_MODE", "LANDAREA_MODE",
    "LIVINGAPARTMENTS_MODE", "LIVINGAREA_MODE",
    "NONLIVINGAPARTMENTS_MODE", "NONLIVINGAREA_MODE",
    "APARTMENTS_MEDI", "BASEMENTAREA_MEDI",
    "YEARS_BEGINEXPLUATATION_MEDI", "YEARS_BUILD_MEDI",
    "COMMONAREA_MEDI", "ELEVATORS_MEDI", "ENTRANCES_MEDI",
    "FLOORSMAX_MEDI", "FLOORSMIN_MEDI", "LANDAREA_MEDI",
    "LIVINGAPARTMENTS_MEDI", "LIVINGAREA_MEDI",
    "NONLIVINGAPARTMENTS_MEDI", "NONLIVINGAREA_MEDI",
    "TOTALAREA_MODE",
    "OBS_30_CNT_SOCIAL_CIRCLE", "DEF_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE",
    "DAYS_LAST_PHONE_CHANGE",
    "AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK", "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_YEAR"
]

# Keep as 0/1; impute if needed
binary_cols = [
    "FLAG_MOBIL", "FLAG_EMP_PHONE", "FLAG_WORK_PHONE",
    "FLAG_CONT_MOBILE", "FLAG_PHONE", "FLAG_EMAIL",
    "REG_REGION_NOT_LIVE_REGION", "REG_REGION_NOT_WORK_REGION",
    "LIVE_REGION_NOT_WORK_REGION", "REG_CITY_NOT_LIVE_CITY",
    "REG_CITY_NOT_WORK_CITY", "LIVE_CITY_NOT_WORK_CITY",
    "FLAG_DOCUMENT_2", "FLAG_DOCUMENT_3", "FLAG_DOCUMENT_4",
    "FLAG_DOCUMENT_5", "FLAG_DOCUMENT_6", "FLAG_DOCUMENT_7",
    "FLAG_DOCUMENT_8", "FLAG_DOCUMENT_9", "FLAG_DOCUMENT_10",
    "FLAG_DOCUMENT_11", "FLAG_DOCUMENT_12", "FLAG_DOCUMENT_13",
    "FLAG_DOCUMENT_14", "FLAG_DOCUMENT_15", "FLAG_DOCUMENT_16",
    "FLAG_DOCUMENT_17", "FLAG_DOCUMENT_18", "FLAG_DOCUMENT_19",
    "FLAG_DOCUMENT_20", "FLAG_DOCUMENT_21"
]

preprocessor = ColumnTransformer([
    ("categorical", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols),

    ("scaled", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), scale_cols),

    ("binary", SimpleImputer(strategy="most_frequent"), binary_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
from scipy.sparse import issparse
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score

if issparse(X_train_processed):
    X_train_processed = X_train_processed.toarray()
    X_test_processed = X_test_processed.toarray()

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32)

print(X_train_tensor.shape)
print("NaNs:", torch.isnan(X_train_tensor).sum().item())

train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=256,
    shuffle=True
)

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

model = NeuralNetwork(X_train_tensor.shape[1])
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(15):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(X_batch)

    print(f"Epoch {epoch + 1}: loss = {total_loss / len(X_train_tensor):.4f}")

model.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(model(X_test_tensor)).numpy()

print("Test ROC AUC:", roc_auc_score(y_test, probabilities))

torch.Size([246008, 248])
NaNs: 0
Epoch 1: loss = 0.2655
Epoch 2: loss = 0.2511
Epoch 3: loss = 0.2504
Epoch 4: loss = 0.2494
Epoch 5: loss = 0.2485
Epoch 6: loss = 0.2476
Epoch 7: loss = 0.2465
Epoch 8: loss = 0.2457
Epoch 9: loss = 0.2448
Epoch 10: loss = 0.2440
Epoch 11: loss = 0.2430
Epoch 12: loss = 0.2421
Epoch 13: loss = 0.2408
Epoch 14: loss = 0.2399
Epoch 15: loss = 0.2390
Test ROC AUC: 0.7385579918319602


##Improved model

In [ ]:
X_train_nn, X_val_nn, y_train_nn, y_val_nn = train_test_split(
    X_train_tensor,
    y_train_tensor,
    test_size = .2,
    random_state = 12,
    stratify = y_train_tensor
)

train_loader = DataLoader(
    TensorDataset(X_train_nn, y_train_nn),
    batch_size = 512,
    shuffle = True
)

val_loader = DataLoader(
    TensorDataset(X_val_nn, y_val_nn),
    batch_size = 1024,
    shuffle = False
)

class NeuralNetwork2(nn.Module):

  def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.30),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(32, 1)
        )

  def foward(self, x):
    return self.network(x).squeeze(1)

model = NeuralNetwork(X_train_tensor.shape[1])
num_positive = y_train_nn.sum()
num_negative = len(y_train_nn) - num_positive
pos_weight = num_negative / num_positive

print("Positive class weight:", pos_weight.item())

Positive class weight: 11.3870849609375


In [ ]:
loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay = 1e-4
    )

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode = "max",
    factor = 0.5,
    patience = 2
)

best_auc = 0
best_state = None
patience = 5
epochs_without_improvement = 0

for epoch in range(20):
  model.train()

  for X_batch, y_batch in train_loader:
    optimizer.zero_grad()
    logits = model(X_batch)
    loss = loss_fn(logits, y_batch)
    loss.backward()
    optimizer.step()
    total_loss += loss.item() * len(X_batch)

  train_loss = total_loss / len(X_train_nn)

  model.eval()

  val_probabilities = []
  val_targets = []

  with torch.no_grad():
    for X_batch, y_batch in val_loader:
      logits = model(X_batch)
      probabilities = torch.sigmoid(logits)

      val_probabilities.extend(
          probabilities.cpu().numpy()
      )

      val_targets.extend(
          y_batch.cpu().numpy()
      )

  val_auc = roc_auc_score(
      val_targets,
      val_probabilities
  )

  scheduler.step(val_auc)

  print(
        f"Epoch {epoch + 1:2d} | "
        f"Loss: {train_loss:.4f} | "
        f"Val AUC: {val_auc:.4f}"
    )

  if val_auc > best_auc:
        best_auc = val_auc

        best_state = {
            key: value.cpu().clone()
            for key, value in model.state_dict().items()
        }

        epochs_without_improvement = 0

  else:
        epochs_without_improvement += 1

  if epochs_without_improvement >= patience:
        print("Early stopping")
        break

Epoch  1 | Loss: 4.7085 | Val AUC: 0.7455
Epoch  2 | Loss: 5.7907 | Val AUC: 0.7459
Epoch  3 | Loss: 6.8679 | Val AUC: 0.7421
Epoch  4 | Loss: 7.9395 | Val AUC: 0.7439
Epoch  5 | Loss: 9.0048 | Val AUC: 0.7449
Epoch  6 | Loss: 10.0572 | Val AUC: 0.7436
Epoch  7 | Loss: 11.1040 | Val AUC: 0.7430
Early stopping


In [ ]:
model.eval()

with torch.no_grad():
    probabilities = torch.sigmoid(
        model(X_test_tensor)
    ).numpy()

test_auc = roc_auc_score(
    y_test_tensor.numpy(),
    probabilities
)

print("Best Validation ROC AUC:", best_auc)
print("Test ROC AUC:", test_auc)

Best Validation ROC AUC: 0.7459423487071449
Test ROC AUC: 0.7471306841833192


##Difference

The improved model has 128 hidden layers compared to 64 for the baseline. It also had other features like batch normalization, dropout and L2 regularzation for stablility and reducing overfitting.